# Day 18. Encoder / Decoder

**오늘의 목표**
- 2017년 원본 Transformer가 **번역**을 위해 Encoder + Decoder 구조로 만들어졌다는 걸 안다
- Encoder(마스크 없음, 양방향)와 Decoder(Causal Mask + Cross-Attention)의 차이를 설명할 수 있다
- **Cross-Attention**을 직접 구현하고, 점수 표가 `(T_dec, T_enc)` **직사각형**이 되는 이유를 안다
- Encoder-only(BERT) / Decoder-only(GPT) / Encoder-Decoder(T5) 세 가족을 구분할 수 있다
- 왜 지금의 LLM은 대부분 Decoder-only인지 설명할 수 있다

> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하세요.
> `# TODO` 셀은 직접 채워보는 연습 문제입니다.

In [ ]:
import math
import hashlib
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(0)

def r(t, d=3):
    # 텐서를 보기 좋게 반올림한 리스트로
    return [round(v, d) for v in t.flatten().tolist()] if t.ndim <= 1 else [[round(v, d) for v in row] for row in t.tolist()]

## 1. 원본 Transformer는 "번역기"였다

2017년 논문 *Attention Is All You Need*의 Transformer는 **기계 번역**을 위해 만들어졌습니다.

```text
입력 (한국어):  나는 학교에 간다
출력 (영어)  :  I go to school
```

번역은 두 가지 일로 나눌 수 있어요.

| 단계 | 하는 일 | 담당 |
|---|---|---|
| ① 읽기 | 한국어 문장 **전체**를 읽고 의미를 파악 | **Encoder** |
| ② 쓰기 | 영어 문장을 **한 단어씩** 생성. 쓸 때마다 한국어 문장을 참고 | **Decoder** |

```text
  나는 학교에 간다                          <s>  I   go  to
        │                                    │
  ┌─────▼──────┐                       ┌─────▼──────────────┐
  │  Encoder   │                       │ Masked Self-Attn   │ ← 지금까지 쓴 영어만 봄
  │ (마스크 X) │                       │        ↓           │
  │            │──── K, V ───────────▶ │ Cross-Attention    │ ← 한국어 문장을 참고
  └────────────┘                       │        ↓           │
                                       │       FFN          │
                                       └─────────┬──────────┘
                                                 ▼
                                           I  go  to  school   (다음 Token 예측)
```

In [ ]:
src_tokens = ["나는", "학교에", "간다"]          # Encoder 입력 (한국어)
tgt_tokens = ["<s>", "I", "go", "to"]            # Decoder 입력 (지금까지 쓴 영어)
tgt_answer = ["I", "go", "to", "school"]         # Decoder가 맞혀야 할 다음 Token

T_enc, T_dec = len(src_tokens), len(tgt_tokens)
print("T_enc =", T_enc, " (한국어 Token 수)")
print("T_dec =", T_dec, " (영어 Token 수)")
print()
for inp, ans in zip(tgt_tokens, tgt_answer):
    print(f"Decoder가 '{inp}'까지 봤을 때 → '{ans}' 예측")

> 💡 `<s>`는 "문장 시작" 특수 Token입니다. (Day 8의 `<|endoftext|>`와 비슷한 역할)
> Decoder 입력과 정답이 **한 칸씩 밀려** 있는 것, Day 6에서 본 x/y 구조와 똑같죠?

## 2. 오늘 쓸 Attention 함수 하나로 세 가지 다 만들기

Day 13·14에서 만든 Attention을 조금 일반화합니다.
**Query를 어디서 가져오고, Key/Value를 어디서 가져오는지**만 다르게 하면 세 종류가 다 나옵니다.

| 종류 | Query 출처 | Key / Value 출처 | Causal Mask |
|---|---|---|---|
| Encoder Self-Attention | Encoder | Encoder | ❌ |
| Decoder Self-Attention | Decoder | Decoder | ✅ |
| **Cross-Attention** | **Decoder** | **Encoder** | ❌ |

In [ ]:
def attention(q, k, v, causal=False):
    # q: (..., T_q, d),  k, v: (..., T_k, d)
    d = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d)          # (..., T_q, T_k)
    if causal:
        T_q, T_k = scores.shape[-2], scores.shape[-1]
        mask = torch.tril(torch.ones(T_q, T_k))
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ v, weights

## 3. Encoder: 문장 전체를 양방향으로 읽는다

Encoder는 입력 문장이 **이미 전부 주어져 있습니다**. 번역할 한국어 문장을 미리 다 알고 있으니까요.
그래서 미래를 가릴 이유가 없고, **Causal Mask를 쓰지 않습니다.**

"나는"도 "간다"를 볼 수 있어요. 이걸 **양방향(bidirectional)** 이라고 부릅니다.

In [ ]:
C = 8
x_enc = torch.randn(T_enc, C)

_, w_enc = attention(x_enc, x_enc, x_enc, causal=False)

print("Encoder Self-Attention Weight  (행 = 지금 Token, 열 = 참고하는 Token)")
print("          " + "".join(f"{t:>8s}" for t in src_tokens))
for t, row in zip(src_tokens, w_enc):
    print(f"{t:>8s}  " + "".join(f"{v.item()*100:7.1f}%" for v in row))
print()
print("0인 칸이 있나?", bool((w_enc == 0).any()), " ← 모든 Token이 모든 Token을 봄")

## 4. Decoder ①: Masked Self-Attention

Decoder는 영어 문장을 **한 단어씩 써 나갑니다**.
"I"를 쓸 때 아직 "go"는 없으니, 학습 때도 미래 Token을 보면 안 됩니다. → **Causal Mask** (Day 14)

In [ ]:
x_dec = torch.randn(T_dec, C)
_, w_dec = attention(x_dec, x_dec, x_dec, causal=True)

print("Decoder Self-Attention Weight")
print("          " + "".join(f"{t:>8s}" for t in tgt_tokens))
for t, row in zip(tgt_tokens, w_dec):
    print(f"{t:>8s}  " + "".join(f"{v.item()*100:7.1f}%" for v in row))

## 5. Decoder ②: Cross-Attention (오늘의 핵심!)

Decoder가 영어를 쓰려면 **한국어 문장을 참고**해야 합니다. 그 통로가 Cross-Attention이에요.

```text
Query        ← Decoder  ("나는 지금 영어 'go'를 쓰려는데, 한국어 어디를 봐야 하지?")
Key, Value   ← Encoder 출력  (한국어 문장의 각 Token 정보)
```

### 5-1. shape 따라가기

In [ ]:
head_size = 16
Wq = nn.Linear(C, head_size, bias=False)
Wk = nn.Linear(C, head_size, bias=False)
Wv = nn.Linear(C, head_size, bias=False)

q = Wq(x_dec)          # Query는 Decoder에서
k = Wk(x_enc)          # Key는 Encoder에서
v = Wv(x_enc)          # Value도 Encoder에서

scores = q @ k.T / math.sqrt(head_size)
weights = F.softmax(scores, dim=-1)
out = weights @ v

print("x_dec  :", tuple(x_dec.shape), "  (T_dec, C)")
print("x_enc  :", tuple(x_enc.shape), "  (T_enc, C)")
print("q      :", tuple(q.shape), " (T_dec, head_size)")
print("k, v   :", tuple(k.shape), " (T_enc, head_size)")
print("scores :", tuple(scores.shape), "  (T_dec, T_enc) ← 정사각형이 아님!")
print("out    :", tuple(out.shape), " (T_dec, head_size) ← Decoder Token 수 그대로")

**관찰 포인트**
- 지금까지 본 Self-Attention 점수 표는 항상 `(T, T)` 정사각형이었죠.
- Cross-Attention은 **행 = 영어 Token(4개), 열 = 한국어 Token(3개)** 라서 `(4, 3)` 직사각형입니다.
- 출력은 **Query 쪽(Decoder) Token 개수**를 따라갑니다. 각 영어 Token이 한국어 정보를 섞어서 가져오는 거예요.
- Cross-Attention에는 **Causal Mask가 없습니다.** 한국어 문장은 처음부터 전부 주어졌으니까요.

### 5-2. 학습된 번역 모델이라면?

위 숫자는 랜덤 Weight라 의미가 없습니다. 잘 학습된 번역 모델의 Cross-Attention은 대략 이런 모양이 됩니다.
(설명을 위해 **손으로 만든 예시 표**예요)

In [ ]:
# 행: 영어 Token이 다음 단어를 쓰려고 할 때 / 열: 한국어 Token
example = torch.tensor([
    # 나는   학교에   간다
    [0.80,  0.05,   0.15],   # <s> → 다음에 'I'를 써야 함   → '나는'
    [0.10,  0.10,   0.80],   # I   → 다음에 'go'           → '간다'
    [0.10,  0.70,   0.20],   # go  → 다음에 'to'           → '학교에' (방향의 '에')
    [0.05,  0.90,   0.05],   # to  → 다음에 'school'       → '학교에'
])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(example, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(T_enc)); ax.set_xticklabels(src_tokens, fontsize=12)
ax.set_yticks(range(T_dec)); ax.set_yticklabels([f"{a} → ({b})" for a, b in zip(tgt_tokens, tgt_answer)], fontsize=11)
for i in range(T_dec):
    for j in range(T_enc):
        ax.text(j, i, f"{example[i, j]:.2f}", ha="center", va="center",
                color="white" if example[i, j] > 0.5 else "black", fontsize=11)
ax.set_xlabel("Key / Value: 한국어 (Encoder)", fontsize=11)
ax.set_ylabel("Query: 영어 (Decoder)", fontsize=11)
ax.set_title("Cross-Attention 예시 (손으로 만든 표)", fontsize=13)
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.show()

영어 단어를 쓸 때마다 **대응하는 한국어 단어**에 주목하고 있죠.
한국어와 영어는 어순이 다른데(`나는 학교에 간다` vs `I go to school`), Cross-Attention이 그 순서 차이를 알아서 맞춰줍니다.
번역 모델에서 이 표를 그려보면 실제로 이런 "정렬(alignment)" 패턴이 나타나요.

## 6. 세 가지 마스크 한눈에 비교

In [ ]:
full_enc   = torch.ones(T_enc, T_enc)
causal_dec = torch.tril(torch.ones(T_dec, T_dec))
full_cross = torch.ones(T_dec, T_enc)

panels = [
    ("Encoder Self-Attention\n(마스크 없음)",   full_enc,   src_tokens, src_tokens),
    ("Decoder Self-Attention\n(Causal Mask)",   causal_dec, tgt_tokens, tgt_tokens),
    ("Cross-Attention\n(마스크 없음, 직사각형)", full_cross, src_tokens, tgt_tokens),
]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, (title, m, cols, rows) in zip(axes, panels):
    ax.imshow(m, cmap="Greens", vmin=0, vmax=1.3)
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, fontsize=11)
    ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows, fontsize=11)
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            ax.text(j, i, "O" if m[i, j] else "X", ha="center", va="center", fontsize=12,
                    color="black" if m[i, j] else "gray")
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Key (보는 대상)")
axes[0].set_ylabel("Query (지금 Token)")
plt.tight_layout()
plt.show()

| | 행(Query) | 열(Key) | shape | 마스크 |
|---|---|---|---|---|
| Encoder Self | 한국어 | 한국어 | `(T_enc, T_enc)` | 없음 |
| Decoder Self | 영어 | 영어 | `(T_dec, T_dec)` | **Causal** |
| Cross | 영어 | 한국어 | `(T_dec, T_enc)` | 없음 |

> 💡 실제 구현에서는 문장 길이를 맞추려고 채운 빈칸(padding)을 가리는 **padding mask**도 쓰지만, 오늘은 생략합니다.

## 7. Encoder Block / Decoder Block 직접 만들기

Day 14의 Multi-Head Attention을 조금 바꿔서, **Key/Value를 다른 곳에서 받을 수 있게** 만듭니다.
LayerNorm과 Residual(`x = x + ...`)은 Day 15·16에서 자세히 배우니, 지금은 "블록을 이렇게 조립한다" 정도로만 보세요.
(GPT-2 이후 쓰는 **Pre-LN** 방식: LayerNorm을 Attention/FFN **앞에** 둠)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, C, n_head, causal=False):
        super().__init__()
        assert C % n_head == 0
        self.n_head, self.causal = n_head, causal
        self.q_proj = nn.Linear(C, C, bias=False)
        self.k_proj = nn.Linear(C, C, bias=False)
        self.v_proj = nn.Linear(C, C, bias=False)
        self.out_proj = nn.Linear(C, C)

    def forward(self, x, context=None):
        # context가 없으면 Self-Attention, 있으면 Cross-Attention
        if context is None:
            context = x
        B, T_q, C = x.shape
        T_k = context.size(1)
        hs = C // self.n_head

        q = self.q_proj(x).view(B, T_q, self.n_head, hs).transpose(1, 2)        # (B, nh, T_q, hs)
        k = self.k_proj(context).view(B, T_k, self.n_head, hs).transpose(1, 2)  # (B, nh, T_k, hs)
        v = self.v_proj(context).view(B, T_k, self.n_head, hs).transpose(1, 2)  # (B, nh, T_k, hs)

        y, _ = attention(q, k, v, causal=self.causal)                            # (B, nh, T_q, hs)
        y = y.transpose(1, 2).contiguous().view(B, T_q, C)                        # (B, T_q, C)
        return self.out_proj(y)


def ffn(C):
    return nn.Sequential(nn.Linear(C, 4 * C), nn.GELU(), nn.Linear(4 * C, C))


class EncoderBlock(nn.Module):
    def __init__(self, C, n_head):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(C), MultiHeadAttention(C, n_head, causal=False)
        self.ln2, self.ffn  = nn.LayerNorm(C), ffn(C)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))          # ① Self-Attention (마스크 없음)
        x = x + self.ffn(self.ln2(x))           # ② FFN
        return x


class DecoderBlock(nn.Module):
    def __init__(self, C, n_head):
        super().__init__()
        self.ln1, self.self_attn  = nn.LayerNorm(C), MultiHeadAttention(C, n_head, causal=True)
        self.ln2, self.cross_attn = nn.LayerNorm(C), MultiHeadAttention(C, n_head, causal=False)
        self.ln3, self.ffn        = nn.LayerNorm(C), ffn(C)

    def forward(self, x, enc_out):
        x = x + self.self_attn(self.ln1(x))                       # ① Masked Self-Attention
        x = x + self.cross_attn(self.ln2(x), context=enc_out)     # ② Cross-Attention (Encoder 참고)
        x = x + self.ffn(self.ln3(x))                             # ③ FFN
        return x

### 7-1. 전체 Encoder-Decoder 모델 조립

In [ ]:
class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, C=32, n_head=4, n_layer=2, block_size=16):
        super().__init__()
        self.src_tok = nn.Embedding(src_vocab, C)
        self.tgt_tok = nn.Embedding(tgt_vocab, C)
        self.pos     = nn.Embedding(block_size, C)
        self.encoder = nn.ModuleList([EncoderBlock(C, n_head) for _ in range(n_layer)])
        self.decoder = nn.ModuleList([DecoderBlock(C, n_head) for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(C)
        self.lm_head = nn.Linear(C, tgt_vocab)

    def encode(self, src):
        T = src.size(1)
        x = self.src_tok(src) + self.pos(torch.arange(T))
        for blk in self.encoder:
            x = blk(x)
        return x

    def decode(self, tgt, enc_out):
        T = tgt.size(1)
        x = self.tgt_tok(tgt) + self.pos(torch.arange(T))
        for blk in self.decoder:
            x = blk(x, enc_out)
        return self.lm_head(self.ln_f(x))

    def forward(self, src, tgt, verbose=False):
        enc_out = self.encode(src)
        logits = self.decode(tgt, enc_out)
        if verbose:
            print("src     :", tuple(src.shape), "    (B, T_enc)")
            print("tgt     :", tuple(tgt.shape), "    (B, T_dec)")
            print("enc_out :", tuple(enc_out.shape), " (B, T_enc, C) ← Decoder의 모든 층에 K/V로 전달")
            print("logits  :", tuple(logits.shape), " (B, T_dec, tgt_vocab)")
        return logits


src_vocab = {w: i for i, w in enumerate(["나는", "학교에", "간다", "집에", "너는"])}
tgt_vocab = {w: i for i, w in enumerate(["<s>", "I", "go", "to", "school", "home", "you"])}

model = Seq2SeqTransformer(len(src_vocab), len(tgt_vocab))
src = torch.tensor([[src_vocab[w] for w in src_tokens]])     # (1, 3)
tgt = torch.tensor([[tgt_vocab[w] for w in tgt_tokens]])     # (1, 4)

logits = model(src, tgt, verbose=True)
print()
print("파라미터 수:", sum(p.numel() for p in model.parameters()))

> 💡 `logits`는 Decoder Token 개수(4)만큼 나옵니다. 각 위치에서 **다음 영어 Token**의 점수예요.
> 학습할 때는 Day 6처럼 `F.cross_entropy(logits.view(-1, V), 정답.view(-1))`로 Loss를 구합니다.

### 7-2. 마스크가 제대로 동작하는지 실험으로 확인

- **Decoder**: 미래 영어 Token을 바꿔도 앞쪽 출력은 그대로여야 함 (Causal)
- **Encoder**: 한국어 Token 하나를 바꾸면 Decoder의 **모든** 위치 출력이 바뀌어야 함 (Cross-Attention은 한국어 전체를 봄)

In [ ]:
model.eval()
with torch.no_grad():
    base = model(src, tgt)

    # ① 마지막 영어 Token('to')을 'home'으로 바꾸기
    tgt2 = tgt.clone(); tgt2[0, -1] = tgt_vocab["home"]
    out2 = model(src, tgt2)
    changed = [(base[0, t] - out2[0, t]).abs().max().item() > 1e-6 for t in range(T_dec)]
    print("마지막 영어 Token 변경 → 위치별 출력이 바뀌었나?")
    for tok, c in zip(tgt_tokens, changed):
        print(f"   {tok:>4s}: {c}")

    # ② 마지막 한국어 Token('간다')을 '집에'로 바꾸기
    src2 = src.clone(); src2[0, -1] = src_vocab["집에"]
    out3 = model(src2, tgt)
    changed = [(base[0, t] - out3[0, t]).abs().max().item() > 1e-6 for t in range(T_dec)]
    print()
    print("마지막 한국어 Token 변경 → 위치별 출력이 바뀌었나?")
    for tok, c in zip(tgt_tokens, changed):
        print(f"   {tok:>4s}: {c}")

- ①: 맨 마지막 위치만 바뀌고 앞쪽은 그대로 → Decoder는 미래를 못 봄 ✅
- ②: `<s>`부터 전부 바뀜 → 영어 첫 단어를 쓸 때도 한국어 문장 **끝**까지 참고함 ✅

## 8. Transformer의 세 가족

원본 Transformer에서 **한쪽만 떼어 쓰는** 모델들이 나왔습니다.

| 가족 | 구조 | 대표 모델 | 학습 방식 | 잘하는 일 |
|---|---|---|---|---|
| **Encoder-only** | Encoder만 (양방향) | BERT, RoBERTa | 문장 중간 빈칸 맞히기 (Masked LM) | 이해: 분류, 검색용 임베딩, 개체명 인식 |
| **Decoder-only** | Decoder만 (Cross-Attention 없음) | GPT 시리즈, LLaMA, Claude 같은 채팅 모델 | 다음 Token 맞히기 | **생성**: 대화, 글쓰기, 코드 |
| **Encoder-Decoder** | 둘 다 + Cross-Attention | 원본 Transformer, T5, BART | 입력 → 출력 변환 | 번역, 요약, 음성 인식 (Whisper) |

```text
BERT의 학습:  "나는 [MASK]에 간다"  →  [MASK] = 학교   (앞뒤를 다 보고 맞힘)
GPT의 학습 :  "나는 학교에"          →  다음 = 간다    (앞만 보고 맞힘)
```

### 8-1. 왜 지금의 LLM은 대부분 Decoder-only일까?

**Decoder-only의 장점**
- **구조가 단순**: Block 하나를 N번 쌓으면 끝. Cross-Attention도, 입력/출력 구분도 없음
- **학습 목표가 하나**: "다음 Token 맞히기". 인터넷의 모든 텍스트가 그대로 학습 데이터가 됨 (정답 라벨이 필요 없음)
- **모든 일을 '이어 쓰기'로**: 번역도 `"영어로 번역: 나는 학교에 간다 →"` 다음을 이어 쓰면 됨. 요약, 질의응답, 코드도 마찬가지
- **규모를 키우기 쉬움**: 크기를 키울수록 성능이 꾸준히 좋아지는 경향이 잘 확인됨

**그렇다고 다른 가족이 사라진 건 아님**
- Encoder-only는 지금도 **검색·추천용 임베딩, 분류**에 많이 쓰임 (작고 빠름, 양방향이라 이해에 유리)
- Encoder-Decoder는 **입력과 출력이 확실히 나뉘는 일**(번역, 음성 → 텍스트)에서 여전히 강함

> 💡 정리: "무엇이든 텍스트를 이어 쓰는 범용 모델"을 만들기에는 Decoder-only가 가장 단순하고 확장하기 좋아서 LLM의 표준이 되었습니다.

## 9. LLM과의 연결: GPT = Decoder에서 Cross-Attention을 뺀 것

오늘 만든 `DecoderBlock`에서 Cross-Attention을 빼면 **GPT의 Block**이 됩니다.

```text
DecoderBlock (원본)              GPT Block
─────────────────────           ─────────────────────
Masked Self-Attention     →     Masked Self-Attention
Cross-Attention           →     (없음: 참고할 Encoder가 없음)
FFN                       →     FFN
```

Encoder가 없으니 "번역할 원문"은 어디에 두냐고요? **그냥 앞쪽 문맥으로 같이 넣습니다.**

```text
Encoder-Decoder:  [Encoder: 나는 학교에 간다]  →  [Decoder: I go to school]
Decoder-only   :  [나는 학교에 간다 → I go to school]   ← 하나의 긴 문장으로 이어 쓰기
```

내일(Day 19)은 이 Decoder-only 구조, 즉 **GPT 전체 모양**을 조립합니다.

## 10. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** 영어 6 Token을 생성 중인 Decoder가, 한국어 9 Token을 읽은 Encoder 출력에 Cross-Attention을 합니다. (B = 2, C = 32, Head 1개, head_size = 32)
실행하기 **전에** shape를 예측하세요.

In [ ]:
scores_shape = "?"   # TODO: Cross-Attention 점수 표 (B, ?, ?)
out_shape    = "?"   # TODO: Cross-Attention 출력 (B, ?, ?)

x_dec = torch.randn(2, 6, 32)
x_enc = torch.randn(2, 9, 32)
_q, _k, _v = x_dec, x_enc, x_enc
_s = _q @ _k.transpose(-2, -1)
_o = F.softmax(_s, dim=-1) @ _v

assert scores_shape == tuple(_s.shape), "점수 표의 행은 Query, 열은 Key 개수예요"
assert out_shape == tuple(_o.shape), "출력은 어느 쪽 Token 개수를 따라갈까요?"
print("Q1 정답!")

**Q2.** Cross-Attention 함수를 완성하세요.
- Query는 `x_dec`에서, Key와 Value는 `x_enc`에서 만듭니다.
- `√head_size`로 나누는 것 잊지 마세요. (Day 13) 마스크는 필요 없습니다.
- 반환값은 출력 하나: `(B, T_dec, head_size)`

In [ ]:
def cross_attention(x_dec, x_enc, Wq, Wk, Wv):
    # x_dec: (B, T_dec, C),  x_enc: (B, T_enc, C),  Wq/Wk/Wv: nn.Linear(C, head_size, bias=False)
    return None  # TODO

torch.manual_seed(42)
C_, hs_ = 16, 8
Wq_ = nn.Linear(C_, hs_, bias=False)
Wk_ = nn.Linear(C_, hs_, bias=False)
Wv_ = nn.Linear(C_, hs_, bias=False)
xd = torch.randn(2, 5, C_)
xe = torch.randn(2, 7, C_)

mine = cross_attention(xd, xe, Wq_, Wk_, Wv_)
assert mine is not None, "함수를 완성해주세요"
assert tuple(mine.shape) == (2, 5, hs_), "출력 shape가 달라요"
ref = F.scaled_dot_product_attention(Wq_(xd), Wk_(xe), Wv_(xe))
assert torch.allclose(mine, ref, atol=1e-5), "값이 달라요. Query/Key/Value 출처와 스케일링을 확인해보세요"
print("Q2 정답!")

**Q3.** 각 모델이 어느 가족인지 적으세요. 보기: `"encoder-only"`, `"decoder-only"`, `"encoder-decoder"`

In [ ]:
families = {
    "BERT":  "?",   # TODO
    "GPT-2": "?",   # TODO
    "T5":    "?",   # TODO
    "LLaMA": "?",   # TODO
}

_key = {"BERT": "e9a22e3065", "GPT-2": "e5f3b2505e", "T5": "97e5906bdc", "LLaMA": "32aa543764"}
_h = lambda s: hashlib.sha256(s.encode()).hexdigest()[:10]
wrong = [name for name, fam in families.items() if _h(f"{name}|{fam}") != _key[name]]
assert not wrong, f"다시 생각해볼 모델: {wrong}  (8절 표를 참고하세요)"
print("Q3 정답!")

**Q4.** 원본 Transformer의 세 가지 Attention 중 **Causal Mask가 필요한 것**은? 각각 `True` / `False`로 적으세요.

In [ ]:
needs_causal_mask = {
    "encoder_self_attention": "?",   # TODO
    "decoder_self_attention": "?",   # TODO
    "cross_attention":        "?",   # TODO
}

_ans = "|".join(str(needs_causal_mask[k]) for k in ["encoder_self_attention", "decoder_self_attention", "cross_attention"])
assert all(isinstance(v, bool) for v in needs_causal_mask.values()), "True 또는 False로 적어주세요"
assert hashlib.sha256(_ans.encode()).hexdigest()[:10] == "b71150ac79", "다시 생각해보세요. '그 정보가 이미 다 주어져 있는가?'가 기준이에요"
print("Q4 정답!")

## 11. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. 원본 Transformer는 어떤 문제를 풀려고 만들어졌고, Encoder와 Decoder는 각각 무슨 일을 하는가?
2. Encoder Self-Attention에는 왜 Causal Mask가 없는가?
3. Cross-Attention에서 Query, Key, Value는 각각 어디서 오는가? 점수 표의 shape가 정사각형이 아닌 이유는?
4. BERT, GPT, T5는 각각 어느 가족이고, 무엇을 잘하는가?
5. GPT Block은 원본 Decoder Block에서 무엇을 뺀 것인가? 왜 뺄 수 있는가?
6. 지금의 LLM이 대부분 Decoder-only인 이유를 두 가지 이상 말해보세요.

**내 답:**

1.
2.
3.
4.
5.
6.